# notebook-assistant — Quickstart

**First time running this?** Execute the install cell below, then **restart the kernel** (Kernel → Restart) before running anything else.

In [1]:
# %pip install -q ipywidgets

In [34]:
import os
from dotenv import load_dotenv

load_dotenv() 

True

In [35]:
# Editable install from the repo root (one directory above this notebook).
# Add [bedrock] if you need AWS Bedrock.
%pip install -q -e ".."
# %pip install -q -e "..[bedrock]"

print("\n→ Install done. If this is the first run, RESTART THE KERNEL now, then skip this cell.")

Note: you may need to restart the kernel to use updated packages.

→ Install done. If this is the first run, RESTART THE KERNEL now, then skip this cell.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.44.2 requires tokenizers<0.20,>=0.19, but you have tokenizers 0.22.2 which is incompatible.


In [36]:
# Fallback: if %pip install failed or you prefer not to install, this makes
# the package importable straight from src/ without installing it.
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import notebook_assistant
print("notebook_assistant version:", notebook_assistant.__version__)

notebook_assistant version: 0.2.0


## 1. Pick a provider

Uncomment the block for your provider. The rest of the notebook works identically regardless of which you pick.

**Never paste API keys into notebook cells.** Set them as environment variables (e.g. in a `.env` file or your shell) and read them with `os.environ`.

In [40]:
import os
from notebook_assistant import NotebookAssistant

# --- OpenAI ---
# assistant = NotebookAssistant(
#     model="openai/gpt-4o",
#     api_key=os.environ["OPENAI_API_KEY"],
# )

# --- Anthropic ---
# assistant = NotebookAssistant(
#     model="anthropic/claude-3-5-sonnet-20241022",
#     api_key=os.environ["ANTHROPIC_API_KEY"],
# )

#--- Google Gemini ---
# assistant = NotebookAssistant(
#     model="gemini/gemini-2.5-flash",
#     api_key=os.environ["GEMINI_API_KEY"],
# )

# --- AWS Bedrock ---
# assistant = NotebookAssistant(
#     model="bedrock/anthropic.claude-3-5-sonnet-20241022-v2:0",
#     aws_region="ap-south-1",
# )

# --- Euron (OpenAI-compatible) ---
assistant = NotebookAssistant(
    model="openai/gpt-4.1-nano",
    api_key=os.environ["EURON_API_KEY"],
    api_base="https://api.euron.one/api/v1/euri",
)

## 2. Ask a single question

In [ ]:
# The assistant refuses to answer from general knowledge (see DEFAULT_SYSTEM_PROMPT),
# so attach minimal context first — then ask.
assistant.add_context(
    instructions=(
        "Last month the site saw: 2.1% conversion rate, 9% return rate, ₹1,480 AOV. "
        "Healthy bands — CVR: 1.8–2.2%, return rate: 7–10%, AOV: ₹1,400–1,600."
    )
)  
assistant.ask("Are any of these metrics outside the healthy band?")

Based on the provided data:

- **Conversion Rate (CVR):** 2.1%
  - Healthy band: 1.8% to 2.2%
  - Current value: 2.1%
  - **Within healthy band**

- **Return Rate:** 9%
  - Healthy band: 7% to 10%
  - Current value: 9%
  - **Within healthy band**

- **Average Order Value (AOV):** ₹1,480
  - Healthy band: ₹1,400 to ₹1,600
  - Current value: ₹1,480
  - **Within healthy band**

**Conclusion:** All three metrics—CVR, return rate, and AOV—are within their respective healthy bands.

---
*Tokens -- input: 185 | output: 159 | total: 344*

## 3. Load DataFrames + instructions

In [30]:
import pandas as pd

# Monthly e-commerce performance for a fashion D2C site.
# April = Big Billion Days style sale event.
df_monthly = pd.DataFrame({
    "month":         ["Jan",     "Feb",     "Mar",     "Apr",     "May",     "Jun"],
    "gmv_inr":       [12_400_000, 14_800_000, 13_200_000, 22_600_000, 15_100_000, 16_200_000],
    "orders":        [8_500,     9_900,     8_700,     16_400,    9_800,     10_500],
    "aov_inr":       [1_459,     1_495,     1_517,     1_378,     1_541,     1_543],
    "sessions":      [420_000,   490_000,   445_000,   780_000,   510_000,   540_000],
    "conversion_rate": [0.0202,  0.0202,    0.0196,    0.0210,    0.0192,    0.0194],
    "return_rate":   [0.08,      0.07,      0.09,      0.15,      0.11,      0.08],
})

assistant.reset()
assistant.add_context(
    instructions="Analyze e-commerce performance across the reported months.",
    dataframes={"monthly_perf": df_monthly},
)

assistant.ask("Which month stands out and why?")

Based on the provided data, April stands out for several reasons:

1. **Highest GMV (Gross Merchandise Value):** April has the highest GMV at 22,600,000 INR, significantly higher than the other months. The next highest GMV is in June at 16,200,000 INR.

2. **Highest Number of Orders:** April also has the highest number of orders at 16,400, which is substantially more than any other month. The second highest is June with 10,500 orders.

3. **Highest Sessions:** April recorded 780,000 sessions, which is the highest among all months. This indicates a higher level of customer engagement or traffic during this month.

4. **Highest Conversion Rate:** April has the highest conversion rate at 0.021, indicating that a higher percentage of sessions resulted in orders compared to other months.

5. **Highest Return Rate:** April also has the highest return rate at 0.15, which might suggest issues with customer satisfaction or product quality, despite the high sales volume.

Overall, April stands out due to its exceptional performance in terms of sales and customer engagement metrics, although the high return rate could be a point of concern.

---
*Tokens -- input: 1004 | output: 245 | total: 1249*

## 4. Add markdown documentation

In [31]:
assistant.reset()
assistant.add_context(
    instructions="Analyze e-commerce performance using the reference documentation's benchmarks and caveats.",
    dataframes={"monthly_perf": df_monthly},
    markdown_path="sample_documentation.md",  # relative to this notebook
)

assistant.ask("Using the benchmarks, flag any months that look abnormal. Call out which caveats apply before drawing conclusions.")

Based on the provided data and benchmarks, let's analyze each metric for any abnormalities and apply the relevant caveats:

### Conversion Rate (CVR)
- **Benchmark:** Healthy band is 1.8% – 2.2%. Flag if < 1.6% or > 2.6%.
- **Data:**
  - Jan: 2.02%
  - Feb: 2.02%
  - Mar: 1.96%
  - Apr: 2.10%
  - May: 1.92%
  - Jun: 1.94%
- **Analysis:** All months fall within the healthy band. No flags.

### Average Order Value (AOV)
- **Benchmark:** Healthy band is ₹1,400 – ₹1,600. Flag if swing > ₹150 MoM.
- **Data:**
  - Jan: ₹1,459
  - Feb: ₹1,495
  - Mar: ₹1,517
  - Apr: ₹1,378
  - May: ₹1,541
  - Jun: ₹1,543
- **Analysis:** 
  - Apr shows a significant drop to ₹1,378 from Mar's ₹1,517, a swing of ₹139. However, this is not greater than ₹150, so it is not flagged.
  - The drop in Apr could be influenced by the "Sale-month distortion" caveat, as sales events can depress AOV due to discounts.

### Return Rate
- **Benchmark:** Healthy band is 7% – 10%. Flag if > 12% sustained for 2+ months.
- **Data:**
  - Jan: 8%
  - Feb: 7%
  - Mar: 9%
  - Apr: 15%
  - May: 11%
  - Jun: 8%
- **Analysis:** 
  - Apr has a return rate of 15%, which is above the healthy band. However, it is not sustained for 2+ months, so it is not flagged for sustained high return rates.
  - The "Return-lag" caveat may apply, as returns are attributed to the shipping month and can be underestimated for recent months.

### GMV MoM Growth
- **Benchmark:** Healthy band is –5% to +15% for non-sale months. Flag if > +30% (likely sale-event driven).
- **Data:**
  - Jan to Feb: +19.35%
  - Feb to Mar: -10.81%
  - Mar to Apr: +71.21%
  - Apr to May: -33.19%
  - May to Jun: +7.28%
- **Analysis:**
  - Apr shows a +71.21% increase from Mar, which is significantly above the +30% threshold, indicating a likely sale event. This aligns with the "Sale-month distortion" caveat.

### Conclusion
- **Apr** is flagged for both a significant GMV increase and a high return rate, likely due to a sale event.
- The drop in AOV in Apr is consistent with the expected impact of sales events.
- No sustained issues with return rates or conversion rates are observed across the months.

---
*Tokens -- input: 1669 | output: 667 | total: 2336*

## 5. Interactive chat

In [32]:
assistant.chat()  # type 'exit' to stop

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [24]:
from notebook_assistant import NotebookAssistant, DEFAULT_SYSTEM_PROMPT

custom_prompt = DEFAULT_SYSTEM_PROMPT + "\nAlways respond in bullet points.\n"

assistant = NotebookAssistant(
    model="openai/gpt-4.1-nano",
    api_key=os.environ["EURON_API_KEY"],
    api_base="https://api.euron.one/api/v1/euri",
    system_prompt=custom_prompt,
)
assistant.add_context(dataframes={"monthly_perf": df_monthly})
assistant.ask("What patterns do you see in conversion rate and return rate?")

- **Conversion Rate:**
  - The conversion rate remains relatively stable across months, with values close to 0.0192 to 0.021.
  - The highest conversion rate is in April at 0.021, and the lowest is in May at 0.0192.
  - The mean conversion rate is approximately 0.01993, indicating consistency.

- **Return Rate:**
  - The return rate shows more variability, ranging from 0.07 to 0.15.
  - The highest return rate occurs in April at 0.15.
  - The lowest return rate is in February at 0.07.
  - The mean return rate is approximately 0.0967, with notable spikes in April.

- **Pattern Insights:**
  - April exhibits both the highest return rate and a high conversion rate, suggesting increased sales may be associated with higher returns.
  - The overall conversion rate is stable, while return rate fluctuates more significantly.
  - There is no clear inverse or direct correlation visible solely from the data, but the spike in return rate in April warrants further analysis to understand causality.

---
*Tokens -- input: 970 | output: 234 | total: 1204*

## 7. Raw text / history

In [33]:
answer = assistant.ask("Summarize in one line.")
print(str(answer))         # plain text
print(answer.usage)         # {'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}
print(assistant.history)    # list of {'role', 'content'} dicts

April is flagged for a significant GMV increase and high return rate, likely due to a sale event, with no sustained issues in other metrics.
{'input_tokens': 2369, 'output_tokens': 29, 'total_tokens': 2398}
[{'role': 'user', 'content': '## REFERENCE DOCUMENTATION\n# E-commerce Performance — Analyst Reference\n\n## Scope\n\n- Vertical: Fashion (apparel + accessories)\n- Channel: Direct-to-consumer website + app, India\n- Currency: INR\n- Reporting period: calendar month\n- All figures are already aggregated — do not recompute from raw orders.\n\n## Metric definitions\n\n| Metric | Definition |\n|---|---|\n| **GMV** | Gross Merchandise Value. Total order value before returns, discounts already applied. |\n| **Orders** | Count of placed orders (one order can contain multiple items). |\n| **AOV** | Average Order Value = GMV / Orders. |\n| **Sessions** | Unique site + app visits. |\n| **CVR** | Conversion rate = Orders / Sessions. |\n| **Return rate** | Items returned / items shipped, attri